# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [24]:
team_setup = {
    'team_name': 'DannyTuran',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Danny Oppenheimer', 'github': 'https://github.com/DannyOppenheimer'},
        {'name': 'TODO', 'github': 'https://github.com/TuranAhmad04'},
    ],
    'repo_url': 'https://github.com/DannyOppenheimer/walmart-hurricane-project',
    'charter_url': 'https://github.com/DannyOppenheimer/walmart-hurricane-project/blob/main/CHARTER.md',
    'notebook_owner_this_week': 'Danny Oppenheimer',
    'meeting_time': 'We live together, no set meeting time',
    'five_worst_problems': ['TODO'],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  FAIL - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  not yet - five specific problems

team setup complete: False


False

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [25]:
import subprocess

REPO_PATH = '../'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

9325bf4 Fix team charter
e8c5a27 Initial commit


In [26]:
import pandas as pd

df = pd.read_csv('../data/walmart_transactions.csv', dtype=str)


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [27]:
decision_log = []

def clean_store_id(df):

    out = df.copy()
    cleaned = (out['store_id']
               .str.strip()
               .str.upper()
               .str.replace(r'^([A-Z]{2})[-_ ]?(\d+)$', r'\1-\2', regex=True))

    decision_log.append({
        'step': 'clean_store_id',
        'decision': "trimmed spaces, uppercased, unified separator to 'XX-###'",
        'rows_touched': int((cleaned != out['store_id']).sum()),
        'rows_dropped': 0,
    })

    out['store_id'] = cleaned
    return out

def drop_duplicates(df):

    out = df.drop_duplicates(keep='first')

    decision_log.append({
        'step': 'drop_duplicates',
        'decision': 'removed exact duplicate rows, kept the first copy',
        'rows_touched': len(df) - len(out),
        'rows_dropped': len(df) - len(out),
    })

    return out

def drop_near_duplicates(df):

    # Same row in every column except transaction_id. Rows with no timestamp are
    # left alone: with the time unknown we cannot say two rows are the same purchase.
    match_cols = [c for c in df.columns if c != 'transaction_id']
    is_copy = df.sort_values('transaction_id').duplicated(subset=match_cols, keep='first')
    is_copy &= df.loc[is_copy.index, 'timestamp'].notna()

    dropped = df.loc[is_copy[is_copy].index, 'transaction_id']
    out = df.drop(index=dropped.index)

    decision_log.append({
        'step': 'drop_near_duplicates',
        'decision': f'dropped rows identical to another row in every column except transaction_id, '
                    f'keeping the lowest ID. The dropped copies have IDs {dropped.min()} to {dropped.max()} '
                    f'(one consecutive block) and differ from the original only by ID, which points to '
                    f're-entered copies. RISK: two shoppers could genuinely have made the same purchase '
                    f'at the same store and second, in which case we undercount by {len(dropped)} rows '
                    f'({len(dropped) / len(df):.2%} of rows). We judge that unlikely and the cleanup defensible.',
        'rows_touched': len(dropped),
        'rows_dropped': len(dropped),
    })

    return out

# The five formats in the raw data
TIMESTAMP_FORMATS = [
    '%Y-%m-%dT%H:%M:%S',
    '%Y-%m-%d %H:%M:%S', 
    '%m/%d/%Y %H:%M',
    '%m-%d-%Y %H:%M:%S',
    '%d-%b-%Y %H:%M',
]

def parse_timestamp(df):

    out = df.copy()
    parsed = pd.Series(pd.NaT, index=out.index, dtype='datetime64[ns]')

    for fmt in TIMESTAMP_FORMATS:
        parsed = parsed.fillna(pd.to_datetime(out['timestamp'], format=fmt, errors='coerce'))

    blank = int(out['timestamp'].isna().sum())
    unparseable = int(parsed.isna().sum()) - blank

    decision_log.append({
        'step': 'parse_timestamp',
        'decision': f'parsed 5 formats to datetime (month-first) - {blank} blank timestamps - {unparseable} unparseable',
        'rows_touched': int(parsed.notna().sum()),
        'rows_dropped': 0,
    })

    out['timestamp'] = parsed
    return out

def clean_price(df):

    out = df.copy()
    has_dollar = out['unit_price'].str.contains('$', regex=False)
    numeric = pd.to_numeric(out['unit_price'].str.replace('$', '', regex=False).str.strip(), errors='coerce')

    unparseable = int(numeric.isna().sum() - out['unit_price'].isna().sum())

    decision_log.append({
        'step': 'clean_price',
        'decision': f"stripped '$' and converted unit_price to float - {unparseable} unparseable",
        'rows_touched': int(has_dollar.sum()),
        'rows_dropped': 0,
    })

    out['unit_price'] = numeric
    return out

CATEGORY_MAP = {
    'alcohol': 'Alcohol',           'alcoholic beverages': 'Alcohol',
    'baby': 'Baby',                 'baby products': 'Baby',
    'breakfast': 'Breakfast',       'breakfast foods': 'Breakfast',
    'electronics': 'Electronics',
    'emergency': 'Emergency',       'emergency supplies': 'Emergency',
    'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'grocery': 'Grocery',           'groceries': 'Grocery',
    'health': 'Health',             'health & wellness': 'Health',
    'household': 'Household',       'household essentials': 'Household',
    'snacks': 'Snacks',
}

def clean_category(df):

    out = df.copy()
    key = out['category'].str.strip().str.lower()
    mapped = key.map(CATEGORY_MAP)

    # A spelling we have not seen is kept as-is
    unmapped = int(mapped.isna().sum() - out['category'].isna().sum())
    mapped = mapped.fillna(out['category'].str.strip())

    decision_log.append({
        'step': 'clean_category',
        'decision': f"merged {out['category'].nunique()} category spellings into {mapped.nunique()} "
                    f"canonical categories - {unmapped} rows with an unmapped spelling",
        'rows_touched': int((mapped != out['category']).sum()),
        'rows_dropped': 0,
    })

    out['category'] = mapped
    return out

def handle_negative_quantity(df):

    out = df.copy()
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')

    negative = out['quantity'] < 0
    out = out[~negative]

    decision_log.append({
        'step': 'handle_negative_quantity',
        'decision': 'dropped rows with quantity < 0 - (if we are trying to analyze purchase behavior, returns are too small to help make decisions, so we can safely drop them)',
        'rows_touched': int(negative.sum()),
        'rows_dropped': int(negative.sum()),
    })

    return out

begin_rows = len(df)

tx = clean_store_id(df)
tx = parse_timestamp(tx)
tx = clean_price(tx)
tx = clean_category(tx)
tx = drop_duplicates(tx)
tx = drop_near_duplicates(tx)
tx = handle_negative_quantity(tx)

end_rows = len(tx)
print(f"Started with {begin_rows} rows, now has {end_rows} (diff of {begin_rows - end_rows})")

Started with 25562 rows, now has 25372 (diff of 190)


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [28]:
pd.set_option('display.max_colwidth', None)

log = pd.DataFrame(decision_log)
log.index = log.index + 1
log.index.name = '#'

display(log)
print('total rows touched:', log['rows_touched'].sum())
print('total rows dropped:', log['rows_dropped'].sum())
print('matches before/after row counts:', log['rows_dropped'].sum() == begin_rows - end_rows)

,step,decision,rows_touched,rows_dropped
#,,,,
1,clean_store_id,"trimmed spaces, uppercased, unified separator to 'XX-###'",4348,0
2,parse_timestamp,parsed 5 formats to datetime (month-first) - 886 blank timestamps - 0 unparseable,24676,0
3,clean_price,stripped '$' and converted unit_price to float - 0 unparseable,3029,0
4,clean_category,merged 37 category spellings into 10 canonical categories - 0 rows with an unmapped spelling,18760,0
5,drop_duplicates,"removed exact duplicate rows, kept the first copy",30,30
6,drop_near_duplicates,"dropped rows identical to another row in every column except transaction_id, keeping the lowest ID. The dropped copies have IDs TXN025503 to TXN025532 (one consecutive block) and differ from the original only by ID, which points to re-entered copies. RISK: two shoppers could genuinely have made the same purchase at the same store and second, in which case we undercount by 30 rows (0.12% of rows). We judge that unlikely and the cleanup defensible.",30,30
7,handle_negative_quantity,"dropped rows with quantity < 0 - (if we are trying to analyze purchase behavior, returns are too small to help make decisions, so we can safely drop them)",130,130


total rows touched: 51003
total rows dropped: 190
matches before/after row counts: True


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [29]:
# TODO

### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [30]:
# TODO

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [31]:
# TODO

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [32]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': False,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': None,        # an integer
    'decisions_logged': None,           # how many entries in your log
    'canonical_products': None,         # distinct products after consolidation
    'weather_api_rows': None,           # rows returned by your API call
    'notebook_runs_clean': False,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  FAIL - team_setup_complete = False
  FAIL - charter_committed = False
PROGRESS (fill in what you have)
  not yet - rows_after_cleaning = None
  not yet - decisions_logged = None
  not yet - canonical_products = None
  not yet - weather_api_rows = None
  not yet - notebook_runs_clean = False

required items complete: False
progress items done: 0 of 5


False

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** _..._

**Plan, with an owner and a date:** _..._

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | not started / in progress / draft answer | _..._ |
| Q2 SKU consolidation | _..._ | _..._ |
| Q3 weather correlation | _..._ | _..._ |
| Q4 store and geography | _..._ | _..._ |
| Q5 signal or artifact | _..._ | _..._ |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.